# N17 — Cardiac Structure Screening from Chest X-ray
## based on Machine Learning and Deep Learning

This Colab notebook uses **classical ML for patient metadata**, **EfficientNet-B0 for chest X-rays**, and a **late-fusion logistic regression model** for the final composite SLVH/DLV probability.

Data policy:

- Every split is created by `patient_id`.
- ML and DL predictions are paired by both `patient_id` and `cxr_filename`.
- `ivsd`, `lvpwd`, and `lvidd` are excluded from ML because the target labels are derived from those echocardiographic measurements.
- The held-out test set is used only for final evaluation.

In [ ]:
import importlib.util
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

required_packages = {
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "joblib": "joblib",
    "torch": "torch",
    "torchvision": "torchvision",
}
missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

print("Environment:", "Google Colab" if IN_COLAB else "local Python")

In [ ]:
from __future__ import annotations

import copy
import json
import random
import zipfile
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

SEED = 42
TARGET = "composite_slvh_dlv"
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
NUM_EPOCHS = 10
PATIENCE = 3
LEARNING_RATE = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Google Drive and dataset discovery

Upload either the complete CheXchoNet ZIP file or the extracted dataset folder to:

`MyDrive/Hoc_may_2_0/DL_Xray/CheXchoNet/`

The notebook extracts the ZIP to Colab temporary storage. Training outputs are saved back to Google Drive.

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_PROJECT = Path("/content/drive/MyDrive/Hoc_may_2_0/DL_Xray/CheXchoNet")
    CACHE_DIR = Path("/content/chexchonet_cache")
else:
    DRIVE_PROJECT = Path.cwd()
    CACHE_DIR = Path.cwd() / "chexchonet_cache"

OUTPUT_DIR = DRIVE_PROJECT / "outputs_ml_support_dl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_dataset_root(search_roots):
    for search_root in map(Path, search_roots):
        if not search_root.exists():
            continue
        direct_candidates = [search_root]
        direct_candidates.extend(path.parent for path in search_root.rglob("metadata.csv"))
        for candidate in direct_candidates:
            if (candidate / "metadata.csv").is_file() and (candidate / "images").is_dir():
                return candidate
    return None


def find_dataset_archive(search_roots):
    candidates = []
    for search_root in map(Path, search_roots):
        if search_root.exists():
            candidates.extend(search_root.glob("*chexchonet*.zip"))
            candidates.extend(search_root.glob("*CheXchoNet*.zip"))
    return max(candidates, key=lambda path: path.stat().st_size) if candidates else None


search_roots = [DRIVE_PROJECT, CACHE_DIR]
DATA_ROOT = find_dataset_root(search_roots)

if DATA_ROOT is None:
    archive_path = find_dataset_archive([DRIVE_PROJECT])
    if archive_path is None:
        raise FileNotFoundError(
            "CheXchoNet was not found. Upload the complete ZIP or extracted folder to "
            f"{DRIVE_PROJECT}"
        )
    try:
        with zipfile.ZipFile(archive_path) as archive:
            names = archive.namelist()
            has_metadata = any(name.endswith("metadata.csv") for name in names)
            has_images = any("/images/" in name.replace("\\", "/") for name in names)
            if not (has_metadata and has_images):
                raise ValueError("The ZIP does not contain metadata.csv and images/")
            CACHE_DIR.mkdir(parents=True, exist_ok=True)
            archive.extractall(CACHE_DIR)
    except zipfile.BadZipFile as exc:
        raise ValueError(
            f"The archive is incomplete or corrupt: {archive_path}. Upload the complete ZIP."
        ) from exc
    DATA_ROOT = find_dataset_root([CACHE_DIR])

if DATA_ROOT is None:
    raise FileNotFoundError("Extraction finished but the CheXchoNet dataset root was not found")

IMAGES_DIR = DATA_ROOT / "images"
print("Dataset root:", DATA_ROOT)
print("Output directory:", OUTPUT_DIR)

## 2. Metadata preparation and leakage controls

The classical ML branch uses only information available outside the echocardiographic definition of the target. Missing transplant or device fields are retained and imputed inside the ML pipeline.

In [ ]:
LABEL_COLUMNS = ("slvh", "dlv", "composite_slvh_dlv")
SAFE_TABULAR_FEATURES = (
    "age_scaled",
    "sex_binary",
    "heart_transplant",
    "lung_transplant",
    "pacemaker_or_icd",
)
LEAKAGE_COLUMNS = ("ivsd", "lvpwd", "lvidd", *LABEL_COLUMNS)
SAFE_BINARY_COLUMNS = ("heart_transplant", "lung_transplant", "pacemaker_or_icd")


def validate_tabular_features(feature_names):
    features = tuple(feature_names)
    leaked = sorted(set(features).intersection(LEAKAGE_COLUMNS))
    if leaked:
        raise ValueError("Features contain target leakage: " + ", ".join(leaked))
    unsupported = sorted(set(features) - set(SAFE_TABULAR_FEATURES))
    if unsupported:
        raise ValueError("Unsupported tabular features: " + ", ".join(unsupported))
    return features


def prepare_metadata(frame):
    required = {"patient_id", "cxr_filename", "age", "sex", *LABEL_COLUMNS}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))

    cleaned = frame.copy()
    cleaned["patient_id"] = cleaned["patient_id"].astype(str).str.strip()
    cleaned["cxr_filename"] = cleaned["cxr_filename"].astype(str).str.strip()
    cleaned["sex"] = cleaned["sex"].astype(str).str.upper().str.strip()
    cleaned["age"] = pd.to_numeric(cleaned["age"], errors="coerce")
    for label in LABEL_COLUMNS:
        cleaned[label] = pd.to_numeric(cleaned[label], errors="coerce")

    valid = (
        cleaned["patient_id"].ne("")
        & cleaned["cxr_filename"].str.lower().str.endswith((".jpg", ".jpeg", ".png"))
        & cleaned["age"].between(18, 90)
        & cleaned["sex"].isin(["F", "M"])
    )
    for label in LABEL_COLUMNS:
        valid &= cleaned[label].isin([0, 1])
    cleaned = cleaned.loc[valid].reset_index(drop=True)

    cleaned["sex_binary"] = cleaned["sex"].map({"F": 0.0, "M": 1.0})
    cleaned["age_scaled"] = (cleaned["age"] - 18.0) / 72.0
    for column in SAFE_BINARY_COLUMNS:
        if column not in cleaned.columns:
            cleaned[column] = np.nan
        else:
            values = pd.to_numeric(cleaned[column], errors="coerce")
            cleaned[column] = values.where(values.isin([0, 1]))
    return cleaned


def safe_stratify(values):
    values = np.asarray(list(values))
    _, counts = np.unique(values, return_counts=True)
    return values if len(counts) > 1 and counts.min() >= 2 else None


def make_patient_split(frame, label=TARGET, seed=SEED):
    patients = frame.groupby("patient_id", as_index=False)[label].max()
    train_patients, holdout = train_test_split(
        patients,
        train_size=0.70,
        random_state=seed,
        stratify=safe_stratify(patients[label]),
    )
    val_patients, test_patients = train_test_split(
        holdout,
        train_size=0.50,
        random_state=seed + 1,
        stratify=safe_stratify(holdout[label]),
    )
    split_map = {
        **dict.fromkeys(train_patients["patient_id"], "train"),
        **dict.fromkeys(val_patients["patient_id"], "val"),
        **dict.fromkeys(test_patients["patient_id"], "test"),
    }
    result = frame.copy()
    result["split"] = result["patient_id"].map(split_map)
    if result["split"].isna().any():
        raise RuntimeError("At least one patient was not assigned to a split")
    if result.groupby("patient_id")["split"].nunique().max() != 1:
        raise RuntimeError("Patient leakage detected across splits")
    return result


def build_tabular_records(frame, label=TARGET, feature_names=SAFE_TABULAR_FEATURES):
    features = validate_tabular_features(feature_names)
    required = {"patient_id", "cxr_filename", "split", label, *features}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError("Missing tabular columns: " + ", ".join(missing))
    keys = ["patient_id", "cxr_filename"]
    if frame.duplicated(keys).any():
        raise ValueError("Duplicate patient_id and cxr_filename keys")
    return frame[[*keys, "split", *features, label]].copy().reset_index(drop=True)


def align_branch_predictions(dl_predictions, ml_predictions):
    keys = ["patient_id", "cxr_filename"]
    for name, frame, probability in (
        ("DL", dl_predictions, "dl_probability"),
        ("ML", ml_predictions, "ml_probability"),
    ):
        missing = sorted(set([*keys, probability]) - set(frame.columns))
        if missing:
            raise ValueError(f"{name} predictions are missing columns: {', '.join(missing)}")
        if frame.duplicated(keys).any():
            raise ValueError(f"Duplicate {name} prediction keys")
    dl_keys = set(map(tuple, dl_predictions[keys].astype(str).to_numpy()))
    ml_keys = set(map(tuple, ml_predictions[keys].astype(str).to_numpy()))
    if dl_keys != ml_keys:
        raise ValueError("ML and DL prediction keys do not match")
    return dl_predictions.merge(
        ml_predictions[[*keys, "ml_probability"]],
        on=keys,
        how="left",
        validate="one_to_one",
        sort=False,
    )

In [ ]:
raw_metadata = pd.read_csv(DATA_ROOT / "metadata.csv")
metadata = prepare_metadata(raw_metadata)
metadata = make_patient_split(metadata)

missing_images = ~metadata["cxr_filename"].map(lambda name: (IMAGES_DIR / name).is_file())
if missing_images.any():
    raise FileNotFoundError(f"Missing {int(missing_images.sum())} image files")

split_audit = (
    metadata.groupby("split")
    .agg(
        images=("cxr_filename", "size"),
        patients=("patient_id", "nunique"),
        positive=(TARGET, "sum"),
        prevalence=(TARGET, "mean"),
    )
    .reset_index()
)
display(split_audit)

manifest_path = OUTPUT_DIR / "metadata_with_patient_split.csv"
metadata.to_csv(manifest_path, index=False)
print("Saved split manifest:", manifest_path)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=split_audit, x="split", y="images", ax=axes[0], color="#0B3C6F")
axes[0].set_title("Images by patient-level split")
sns.barplot(data=split_audit, x="split", y="prevalence", ax=axes[1], color="#C8102E")
axes[1].set_title("Composite label prevalence")
axes[1].set_ylim(0, max(0.20, split_audit["prevalence"].max() * 1.2))
plt.tight_layout()
plt.show()

## 3. Classical ML branch for patient metadata

Two candidates are compared on the validation set. Logistic regression provides an interpretable linear baseline. HistGradientBoosting captures nonlinear relationships. Both receive only the approved metadata predictors.

In [ ]:
FEATURES = validate_tabular_features(SAFE_TABULAR_FEATURES)
tabular = build_tabular_records(metadata)

train_rows = tabular[tabular["split"].eq("train")].reset_index(drop=True)
val_rows = tabular[tabular["split"].eq("val")].reset_index(drop=True)
test_rows = tabular[tabular["split"].eq("test")].reset_index(drop=True)

numeric_preprocessor = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
tree_preprocessor = Pipeline([("imputer", SimpleImputer(strategy="median"))])

ml_candidates = {
    "Logistic Regression": Pipeline(
        [
            ("preprocess", numeric_preprocessor),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "HistGradientBoosting": Pipeline(
        [
            ("preprocess", tree_preprocessor),
            (
                "model",
                HistGradientBoostingClassifier(
                    learning_rate=0.05,
                    max_iter=250,
                    l2_regularization=1.0,
                    random_state=SEED,
                ),
            ),
        ]
    ),
}


def select_threshold(y_true, probability, beta=2.0):
    best_threshold = 0.50
    best_score = -1.0
    for threshold in np.linspace(0.05, 0.95, 181):
        score = fbeta_score(y_true, probability >= threshold, beta=beta, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
    return best_threshold, float(best_score)


def metric_row(name, y_true, probability, threshold):
    prediction = probability >= threshold
    return {
        "model": name,
        "threshold": float(threshold),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f2": fbeta_score(y_true, prediction, beta=2.0, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probability),
        "pr_auc": average_precision_score(y_true, probability),
    }


X_train, y_train = train_rows[list(FEATURES)], train_rows[TARGET].astype(int)
X_val, y_val = val_rows[list(FEATURES)], val_rows[TARGET].astype(int)

ml_validation_rows = []
fitted_candidates = {}
for name, candidate in ml_candidates.items():
    if name == "HistGradientBoosting":
        class_counts = y_train.value_counts()
        class_weight = {
            label: len(y_train) / (2.0 * count)
            for label, count in class_counts.items()
        }
        sample_weight = y_train.map(class_weight).to_numpy()
        candidate.fit(X_train, y_train, model__sample_weight=sample_weight)
    else:
        candidate.fit(X_train, y_train)
    probability = candidate.predict_proba(X_val)[:, 1]
    threshold, _ = select_threshold(y_val, probability)
    ml_validation_rows.append(metric_row(name, y_val, probability, threshold))
    fitted_candidates[name] = (candidate, threshold)

ml_validation = pd.DataFrame(ml_validation_rows).sort_values("pr_auc", ascending=False)
display(ml_validation)

BEST_ML_NAME = ml_validation.iloc[0]["model"]
best_ml_model, best_ml_threshold = fitted_candidates[BEST_ML_NAME]
print("Selected ML model:", BEST_ML_NAME)


def predict_ml(frame):
    probability = best_ml_model.predict_proba(frame[list(FEATURES)])[:, 1]
    return frame[["patient_id", "cxr_filename", TARGET]].rename(
        columns={TARGET: "target"}
    ).assign(ml_probability=probability)


ml_val_predictions = predict_ml(val_rows)
ml_test_predictions = predict_ml(test_rows)
joblib.dump(best_ml_model, OUTPUT_DIR / "metadata_ml_model.joblib")

## 4. Deep learning branch for chest X-rays

EfficientNet-B0 receives only the image. Image augmentation is applied to training data only. Class imbalance is handled with `pos_weight` in binary cross-entropy.

In [ ]:
train_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),
        transforms.RandomRotation(5),
        transforms.ColorJitter(brightness=0.10, contrast=0.10),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=EfficientNet_B0_Weights.DEFAULT.transforms().mean,
            std=EfficientNet_B0_Weights.DEFAULT.transforms().std,
        ),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=EfficientNet_B0_Weights.DEFAULT.transforms().mean,
            std=EfficientNet_B0_Weights.DEFAULT.transforms().std,
        ),
    ]
)


class CheXchoNetImageDataset(Dataset):
    def __init__(self, frame, images_dir, transform):
        self.frame = frame.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(self.images_dir / row["cxr_filename"]).convert("RGB")
        image = self.transform(image)
        return (
            image,
            np.float32(row[TARGET]),
            str(row["patient_id"]),
            str(row["cxr_filename"]),
        )


train_frame = metadata[metadata["split"].eq("train")].reset_index(drop=True)
val_frame = metadata[metadata["split"].eq("val")].reset_index(drop=True)
test_frame = metadata[metadata["split"].eq("test")].reset_index(drop=True)

train_loader = DataLoader(
    CheXchoNetImageDataset(train_frame, IMAGES_DIR, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    CheXchoNetImageDataset(val_frame, IMAGES_DIR, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    CheXchoNetImageDataset(test_frame, IMAGES_DIR, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
def build_image_model():
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    input_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(input_features, 1)
    return model


image_model = build_image_model().to(DEVICE)
positive_count = int(train_frame[TARGET].sum())
negative_count = len(train_frame) - positive_count
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([negative_count / positive_count], device=DEVICE)
)
optimizer = torch.optim.AdamW(image_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


def train_image_epoch(model, loader):
    model.train()
    total_loss = 0.0
    for images, targets, _, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True).view(-1, 1)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += float(loss.item()) * len(images)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_dl(model, loader):
    model.eval()
    rows = []
    for images, targets, patient_ids, filenames in loader:
        probability = torch.sigmoid(model(images.to(DEVICE, non_blocking=True))).squeeze(1)
        for patient_id, filename, target, prob in zip(
            patient_ids,
            filenames,
            targets.numpy(),
            probability.cpu().numpy(),
        ):
            rows.append(
                {
                    "patient_id": str(patient_id),
                    "cxr_filename": str(filename),
                    "target": int(target),
                    "dl_probability": float(prob),
                }
            )
    return pd.DataFrame(rows)


best_state = None
best_pr_auc = -np.inf
epochs_without_improvement = 0
training_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_image_epoch(image_model, train_loader)
    dl_val_predictions = predict_dl(image_model, val_loader)
    val_pr_auc = average_precision_score(
        dl_val_predictions["target"], dl_val_predictions["dl_probability"]
    )
    training_history.append(
        {"epoch": epoch, "train_loss": train_loss, "val_pr_auc": val_pr_auc}
    )
    print(f"Epoch {epoch:02d}: loss={train_loss:.4f}, val PR-AUC={val_pr_auc:.4f}")
    if val_pr_auc > best_pr_auc:
        best_pr_auc = val_pr_auc
        best_state = copy.deepcopy(image_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping")
            break

if best_state is None:
    raise RuntimeError("Image model training did not produce a checkpoint")
image_model.load_state_dict(best_state)
torch.save(best_state, OUTPUT_DIR / "efficientnet_b0_xray.pt")
pd.DataFrame(training_history).to_csv(OUTPUT_DIR / "dl_training_history.csv", index=False)

dl_val_predictions = predict_dl(image_model, val_loader)
dl_test_predictions = predict_dl(image_model, test_loader)

## 5. ML-supported DL fusion

The fusion model receives two values for the same X-ray encounter: the metadata ML probability and the image DL probability. Logistic regression learns how much weight to assign to each branch. The validation set fits the fusion model and selects thresholds; the test set remains untouched until the following cell.

In [ ]:
fusion_val = align_branch_predictions(dl_val_predictions, ml_val_predictions)
fusion_test = align_branch_predictions(dl_test_predictions, ml_test_predictions)

fusion_features = ["ml_probability", "dl_probability"]
fusion_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=SEED,
            ),
        ),
    ]
)
fusion_model.fit(fusion_val[fusion_features], fusion_val["target"])

fusion_val_probability = fusion_model.predict_proba(fusion_val[fusion_features])[:, 1]
fusion_threshold, fusion_val_f2 = select_threshold(
    fusion_val["target"], fusion_val_probability
)
fusion_test_probability = fusion_model.predict_proba(fusion_test[fusion_features])[:, 1]
fusion_test["fusion_probability"] = fusion_test_probability

dl_threshold, _ = select_threshold(
    dl_val_predictions["target"], dl_val_predictions["dl_probability"]
)

test_metrics = pd.DataFrame(
    [
        metric_row(
            f"ML: {BEST_ML_NAME}",
            fusion_test["target"],
            fusion_test["ml_probability"],
            best_ml_threshold,
        ),
        metric_row(
            "DL: EfficientNet-B0",
            fusion_test["target"],
            fusion_test["dl_probability"],
            dl_threshold,
        ),
        metric_row(
            "Fusion: ML supports DL",
            fusion_test["target"],
            fusion_test["fusion_probability"],
            fusion_threshold,
        ),
    ]
)
display(test_metrics)

fusion_prediction = fusion_test_probability >= fusion_threshold
matrix = confusion_matrix(fusion_test["target"], fusion_prediction)
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Fusion model confusion matrix")
plt.show()

fraction_positive, mean_predicted = calibration_curve(
    fusion_test["target"], fusion_test_probability, n_bins=10, strategy="quantile"
)
plt.plot(mean_predicted, fraction_positive, marker="o", label="Fusion")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Ideal")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed positive fraction")
plt.title("Fusion probability calibration")
plt.legend()
plt.show()

joblib.dump(fusion_model, OUTPUT_DIR / "fusion_logistic_regression.joblib")
test_metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
fusion_test.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
with (OUTPUT_DIR / "selected_thresholds.json").open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "ml_model": BEST_ML_NAME,
            "ml_threshold": best_ml_threshold,
            "dl_threshold": dl_threshold,
            "fusion_threshold": fusion_threshold,
        },
        handle,
        indent=2,
    )

print("Saved all models and results to:", OUTPUT_DIR)

## 6. Interpretation and reporting

For the Week 5–6 report, compare the three test rows: metadata ML, image DL, and fusion. The fusion model counts as an improvement only if it raises clinically relevant recall/F2 or improves calibration without an unacceptable precision loss.

Limitations:

- Metadata is limited to age, sex, transplant history, and pacemaker/ICD status. It is not a full laboratory record.
- Several images can belong to one patient, so the split must remain patient-based.
- CheXchoNet supports screening for structural cardiac findings associated with SLVH/DLV. It does not directly diagnose every cardiovascular disease.